# Pipeline di Estrazione e Preprocessing Dati - Head & Neck Cancer (No Leakage)

Questo notebook documenta la complessa pipeline di ingegnerizzazione dei dati che trasforma i raw data disaggregati (clinici, di follow-up e fattori di rischio) in un dataset strutturato e pronto per l'addestramento dei modelli di Machine Learning per l'analisi di sopravvivenza.

**Obiettivo principale della pipeline:** Creare un dataset "Tempo 0 purissimo", eliminando ogni forma di Data Leakage temporale (es. righe relative a recidive o metastasi successive) per garantire metriche di valutazione oneste e clinicamente rilevanti.

**Input** (nella cartella `data/`): `oral_cancer_dataset.xlsx`, `follow_up.xlsx`, `fattori_di_rischio.xlsx`.

**Output:** `dataset.csv`, con una riga per paziente: 527 pazienti × 35 colonne (33 feature, il tempo di follow-up e l'indicatore di evento).

I percorsi dei file sono relativi: il notebook va eseguito dalla cartella `data/`.

---

### Fase 1: Caricamento Dati Raw
Il processo inizia importando i tre file Excel originali contenenti le informazioni disgiunte del progetto TCGA-HNSC:
* **Dati Clinici:** Informazioni anagrafiche, diagnosi e trattamenti.
* **Follow-up:** Tempistiche di contatto e stato in vita.
* **Fattori di Rischio:** Dati relativi all'esposizione a tabacco e alcol.
Tutti gli ID dei pazienti (`case_id`) vengono normalizzati (spazi rimossi, convertiti in maiuscolo) per garantire un matching perfetto nelle fasi successive.

### Fase 2: Aggregazione Dati Clinici per Paziente (Tempo 0 Purissimo)
Questa è la fase cruciale per prevenire il Data Leakage. La pipeline applica un filtro rigoroso per isolare **esclusivamente le righe relative al tumore primario (`primary`)**, scartando a priori qualsiasi record classificato come `recurrence` o `metastasis`.
Poiché un paziente può avere record multipli al Tempo 0 (es. a causa di trattamenti successivi pianificati), i dati vengono aggregati logicamente:
* **Dati Statici e Stadiazione TNM:** Viene estratto il primo valore valido per ogni paziente. Per il TNM, viene data sempre priorità allo stadio *patologico*; in sua assenza o in caso di incompletezza (es. valori TX, NX), il sistema fa un fallback automatico sul TNM *clinico*.
* **Trattamenti Pianificati (Trt):** Vengono raggruppati in macro-categorie binarie (Farmacologico, Radioterapico, Chirurgico), unendo le informazioni di tutte le righe del tumore primario.
* **Intento Clinico (Intent):** Vengono codificate le finalità terapeutiche stabilite alla diagnosi (Curativo, Adiuvante, Palliativo, ecc.).

### Fase 3: Follow-up (Filtro per definire il Time della Survival)
Essendo il nostro obiettivo l'analisi di sopravvivenza, la variabile temporale è cruciale. Filtriamo il dataframe di follow-up isolando esclusivamente i record etichettati come "Last Contact", estraendo così il numero definitivo di giorni di osservazione validi (`days_to_follow_up`) per ciascun paziente.

### Fase 4: Fattori di Rischio e Gestione Missing
* **Smokeless Tobacco:** per i pazienti con almeno un record di esposizione a *Smokeless Tobacco*, i dati del modulo fattori di rischio vengono scartati, perché il loro profilo di esposizione non è comparabile a quello del resto della coorte. Poiché il merge finale è di tipo `left` sul blocco clinico, questi pazienti restano nel dataset con le variabili di esposizione mancanti.
* **Imputazione Deduttiva:** Per i pazienti classificati come "Lifelong Non-Smoker", il valore dei *pack years* viene forzato a `0.0` (deduzione logica inconfutabile).
* **Missing Value Preservation:** Tutti gli altri valori numerici mancanti (es. giorni di alcol a settimana) vengono deliberatamente lasciati come `NaN`, delegando l'imputazione avanzata all'algoritmo multivariato MICE nelle pipeline dei modelli a valle.

### Fase 5: Merge Globale ed Encoding Categoriche
I tre set di dati aggregati vengono uniti in un unico Dataframe. Segue una massiccia fase di Feature Engineering per preparare le variabili ai modelli ML:
* **Creazione Target Survival:** La colonna dello stato vitale viene binarizzata (`1 = DEAD`, `0 = ALIVE` per censurati).
* **Codifica One-Hot:** Variabili nominali come Sede del Tumore (raggruppata in macro-aree anatomiche), Razza (raggruppata in tre categorie), Etnia e Lateralità vengono trasformate in colonne *dummy*. Per Razza, Etnia e Lateralità i valori mancanti ottengono una colonna dedicata (`_Missing`).
* **Smoking Status Ordinale:** Lo status di fumatore viene mappato su una scala ordinale che riflette la gravità e la durata dell'esposizione (da Non-fumatore a Fumatore Attivo). Lo stato non noto viene codificato con `-1`.
* **Encoding TNM Granulare:** I codici TNM vengono convertiti in valori ordinali numerici, prestando attenzione a differenziare stadi clinicamente distinti (ad esempio, T4A vs T4B). I valori originali mancanti o ambigui vengono etichettati con `-1`.

### Fase 6: Pulizia Finale e Output
Vengono rimosse le colonne di servizio ormai inutili (come l'ID del paziente e la risposta alla malattia). Il sistema esegue un controllo di qualità fondamentale eliminando gli eventuali pazienti rimasti privi di un dato temporale di follow-up valido (essenziale per i modelli di Survival). Il dataset finale pulito, privo di leakage temporale, viene esportato in formato CSV (`dataset.csv`, 527 pazienti × 35 colonne), pronto per l'addestramento algoritmico. I placeholder `-1` e i valori `NaN` vengono gestiti a valle, nei notebook dei singoli modelli.

In [1]:
import pandas as pd
import numpy as np
import math
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURAZIONE
# ─────────────────────────────────────────────────────────────────────────────
FILE_ORAL = "oral_cancer_dataset.xlsx"
FILE_FOLLOW = "follow_up.xlsx"
FILE_RISK = "fattori_di_rischio.xlsx"
FILE_OUTPUT = "dataset.csv"

ID = 'cases.case_id'

print("═" * 65)
print("  PIPELINE FROM RAW — oral cancer TCGA-HNSC (NO LEAKAGE)")
print("═" * 65)

# ─────────────────────────────────────────────────────────────────────────────
# 1. CARICAMENTO
# ─────────────────────────────────────────────────────────────────────────────
print("\n[1/6] Caricamento file raw...")

oral = pd.read_excel(FILE_ORAL)
follow = pd.read_excel(FILE_FOLLOW)
risk = pd.read_excel(FILE_RISK)

for df in [oral, follow, risk]:
    df[ID] = df[ID].astype(str).str.strip().str.upper()

print(f"    Clinico: {oral.shape} | Follow-up: {follow.shape} | Rischio: {risk.shape}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. AGGREGAZIONE DATI CLINICI PER PAZIENTE (TEMPO 0 PURISSIMO)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[2/6] Aggregazione per paziente al Tempo 0...")

EXCL_GEN = {"'--", '--', 'nan', 'none', 'null', '', 'not reported', 'unknown'}
EXCL_TNM = EXCL_GEN | {'tx', 'nx', 'mx'}


# Primo valore valido di una serie, in maiuscolo; None se sono tutti mancanti o placeholder.
def first_valid(series, excl=None):
    if excl is None:
        excl = EXCL_GEN
    vals = series.dropna().astype(str)
    vals = vals[~vals.str.strip().str.lower().isin(excl)]
    return vals.iloc[0].strip().upper() if not vals.empty else None


# Stadio TNM del paziente: priorità al valore patologico, con fallback su quello clinico.
def get_tnm(group, path_col, clin_col):
    return first_valid(group[path_col], EXCL_TNM) or \
        first_valid(group[clin_col], EXCL_TNM)


PHARMA_TYPES = {
    'Chemotherapy', 'Pharmaceutical Therapy, NOS',
    'Immunotherapy (Including Vaccines)', 'Targeted Molecular Therapy'
}
RAD_TYPES = {
    'Radiation Therapy, NOS', 'Radiation, Cyberknife',
    'Radiation, External Beam', 'Radiation, Implants',
    'Radiation, Stereotactic/Gamma Knife/SRS'
}

EXCL_TRT = {"'--", '--', 'nan', 'none', 'null', '', 'not reported', 'unknown'}

# FILTRO FONDAMENTALE: Teniamo ESCLUSIVAMENTE le righe relative al tumore primario (Tempo 0)
# Qualsiasi riga marcata come 'recurrence' o 'metastasis' viene distrutta qui.
oral_primary = oral[oral['diagnoses.classification_of_tumor'].astype(str).str.strip().str.lower() == 'primary'].copy()

agg_rows = []

for case_id, group in oral_primary.groupby(ID, sort=True):
    primary_site = first_valid(group['cases.primary_site'])
    age = first_valid(group['demographic.age_at_index'])
    ethnicity = first_valid(group['demographic.ethnicity'])
    gender = first_valid(group['gender'])
    race = first_valid(group['demographic.race'])
    vital = first_valid(group['demographic.vital_status'])  # Il nostro vero Target (Vivo/Morto)
    laterality = first_valid(group['diagnoses.laterality'])

    final_t = get_tnm(group, 'diagnoses.ajcc_pathologic_t', 'diagnoses.ajcc_clinical_t')
    final_n = get_tnm(group, 'diagnoses.ajcc_pathologic_n', 'diagnoses.ajcc_clinical_n')
    final_m = get_tnm(group, 'diagnoses.ajcc_pathologic_m', 'diagnoses.ajcc_clinical_m')

    # Estrazione Trattamenti pianificati al tempo 0
    grp_yes = group[group['treatments.treatment_or_therapy'].astype(str).str.lower() == 'yes']
    trt_vals = set(grp_yes['treatments.treatment_type'].dropna().astype(str)) - EXCL_TRT

    # Estrazione Intento terapeutico al tempo 0
    grp_active = group[group['treatments.treatment_or_therapy'].astype(str).str.lower().isin(['yes', 'unknown'])]
    intent_vals = set(grp_active['treatments.treatment_intent_type'].dropna().astype(str))
    intent_vals = {v for v in intent_vals if v.lower().strip() not in EXCL_TRT}

    agg_rows.append({
        ID: case_id,
        'cases.primary_site': primary_site,
        'demographic.age_at_index': age,
        'demographic.ethnicity': ethnicity,
        'gender': gender,
        'demographic.race': race,
        'demographic.vital_status': vital,
        'diagnoses.laterality': laterality,
        'Final_T': final_t,
        'Final_N': final_n,
        'Final_M': final_m,
        'Trt_PHARMACEUTICAL': int(bool(PHARMA_TYPES & trt_vals)),
        'Trt_RADIATION': int(bool(RAD_TYPES & trt_vals)),
        'Trt_SURGERY': int('Surgery, NOS' in trt_vals),
        'Intent_ADJUVANT': int('Adjuvant' in intent_vals),
        'Intent_CURE': int('Cure' in intent_vals),
        'Intent_BEFORETRT': int(bool({'Induction', 'Neoadjuvant'} & intent_vals)),
        'Intent_PALLIATIVE': int('Palliative' in intent_vals),
    })

df = pd.DataFrame(agg_rows)
print(f"    Pazienti analizzati al Tempo 0: {len(df)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. FOLLOW-UP (Filtro per definire il Time della Survival)
# ─────────────────────────────────────────────────────────────────────────────
print("\n[3/6] Follow-up: estrazione tempo di sopravvivenza...")

df_fup = follow.copy()
df_fup['follow_ups.timepoint_category'] = (
    df_fup['follow_ups.timepoint_category'].astype(str).str.strip().str.title()
)
df_fup_last = df_fup[df_fup['follow_ups.timepoint_category'] == 'Last Contact'].copy()
df_fup_last['follow_ups.days_to_follow_up'] = pd.to_numeric(
    df_fup_last['follow_ups.days_to_follow_up'], errors='coerce'
)
# Una sola osservazione per paziente: il tempo registrato all'ultimo contatto.
df_fup_clean = (
    df_fup_last.groupby(ID)['follow_ups.days_to_follow_up']
    .first().reset_index()
)

# ─────────────────────────────────────────────────────────────────────────────
# 4. FATTORI DI RISCHIO
# ─────────────────────────────────────────────────────────────────────────────
print("\n[4/6] Integrazione Fattori di Rischio...")

# I record di esposizione dei pazienti con almeno una riga 'Smokeless Tobacco' vengono scartati:
# dopo il merge left questi pazienti restano nel dataset con le esposizioni mancanti.
smokeless_ids = set(risk[risk['exposures.exposure_type'] == 'Smokeless Tobacco'][ID])
risk_c = risk[~risk[ID].isin(smokeless_ids)].copy()

risk_c = risk_c.replace(["'--", 'Not Reported', 'Unknown', 'unknown'], np.nan)
COLS_NUM = [
    'exposures.alcohol_days_per_week',
    'exposures.alcohol_drinks_per_day',
    'exposures.pack_years_smoked'
]
for col in COLS_NUM:
    risk_c[col] = pd.to_numeric(risk_c[col], errors='coerce')

risk_clean = (
    risk_c[[ID, 'exposures.tobacco_smoking_status'] + COLS_NUM]
    .groupby(ID).first().reset_index()
)

# Imputazione deduttiva: chi non ha mai fumato ha 0 pack-years.
risk_clean.loc[
    risk_clean['exposures.tobacco_smoking_status'] == 'Lifelong Non-Smoker',
    'exposures.pack_years_smoked'
] = 0.0

# ─────────────────────────────────────────────────────────────────────────────
# 5. MERGE GLOBALE E ENCODING CATEGORICHE
# ─────────────────────────────────────────────────────────────────────────────
print("\n[5/6] Merge e Feature Encoding...")

df = pd.merge(df, df_fup_clean, on=ID, how='left')
df = pd.merge(df, risk_clean, on=ID, how='left')

STR_COLS = [
    'demographic.vital_status', 'gender', 'cases.primary_site',
    'demographic.race', 'demographic.ethnicity', 'diagnoses.laterality',
    'Final_T', 'Final_N', 'Final_M', 'exposures.tobacco_smoking_status'
]
for col in STR_COLS:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace(['NAN', 'NONE', 'NULL', 'N/A', 'NA', "'--", '--'], np.nan)

# CREAZIONE TARGET PER SURVIVAL (1 = Morto, 0 = Vivo/Censurato)
df['demographic.vital_status_DEAD'] = (df['demographic.vital_status'] == 'DEAD').astype(int)
df['gender_MALE'] = (df['gender'] == 'MALE').astype(int)
df = df.drop(columns=['demographic.vital_status', 'gender'])


# Raggruppa la sede anatomica del tumore in sei macro-aree.
def group_site(val):
    s = str(val)
    if 'FLOOR OF MOUTH' in s: return 'FLOOR OF MOUTH'
    if 'TONGUE' in s: return 'TONGUE'
    if 'LARYNX' in s or 'HYPOPHARYNX' in s: return 'LARYNX & HYPOPHARYNX'
    if 'TONSIL' in s or 'OROPHARYNX' in s: return 'OROPHARYNX & TONSIL'
    if 'LIP' in s: return 'LIP'
    return 'OTHER ORAL CAVITY'


df['cases.primary_site'] = df['cases.primary_site'].apply(group_site)
dummies_site = pd.get_dummies(
    df['cases.primary_site'], prefix='cases.primary_site', dummy_na=False, dtype=int
)
df = pd.concat([df.drop(columns=['cases.primary_site']), dummies_site], axis=1)


# Raggruppa la razza in tre categorie (WHITE, ASIAN, NOT WHITE); i mancanti restano NaN.
def group_race(val):
    if pd.isna(val): return np.nan
    s = str(val)
    if 'WHITE' in s: return 'WHITE'
    if 'ASIAN' in s: return 'ASIAN'
    return 'NOT WHITE'


df['demographic.race'] = df['demographic.race'].apply(group_race)

# Scala ordinale dell'esposizione al fumo; lo stato non noto viene codificato con -1.
SMOKING_ORDER = {
    'LIFELONG NON-SMOKER': 0,
    'CURRENT REFORMED SMOKER FOR > 15 YRS': 1,
    'CURRENT REFORMED SMOKER FOR < OR = 15 YRS': 2,
    'CURRENT SMOKER': 3,
}
df['exposures.tobacco_smoking_status_enc'] = (
    df['exposures.tobacco_smoking_status']
    .map(SMOKING_ORDER)
    .fillna(-1).astype(int)
)
df = df.drop(columns=['exposures.tobacco_smoking_status'])

# Codifica ordinale del TNM; i valori mancanti o non mappati diventano -1.
T_MAP = {'TIS': 0, 'T0': 1, 'T1': 2, 'T2': 3, 'T3': 4, 'T4': 5, 'T4A': 6, 'T4B': 7}
N_MAP = {'N0': 0, 'N1': 1, 'N1B': 2, 'N2': 3, 'N2A': 4, 'N2B': 5, 'N2C': 6, 'N3': 7}
M_MAP = {'M0': 0, 'M1': 1}

df['Final_T_enc'] = df['Final_T'].map(T_MAP).fillna(-1).astype(int)
df['Final_N_enc'] = df['Final_N'].map(N_MAP).fillna(-1).astype(int)
df['Final_M_enc'] = df['Final_M'].map(M_MAP).fillna(-1).astype(int)
df = df.drop(columns=['Final_T', 'Final_N', 'Final_M'])

# One-hot delle variabili nominali, con una colonna dedicata ai valori mancanti (_Missing).
NOMINAL_COLS = ['diagnoses.laterality', 'demographic.race', 'demographic.ethnicity']
df = pd.get_dummies(df, columns=NOMINAL_COLS, dummy_na=True, dtype=int)
df.columns = df.columns.str.replace('_nan', '_Missing', regex=False)

# ─────────────────────────────────────────────────────────────────────────────
# 6. PULIZIA FINALE E OUTPUT
# ─────────────────────────────────────────────────────────────────────────────
print("\n[6/6] Pulizia finale e salvataggio...")

# Rimuoviamo colonne inutili (ID non serve al modello predittivo)
df = df.drop(columns=[ID, 'follow_ups.disease_response'], errors='ignore')

# Drop dei pazienti senza una variabile TIME valida (essenziale per la Survival Analysis)
n_before = len(df)
df = df[df['follow_ups.days_to_follow_up'].notna()].reset_index(drop=True)
print(f"    Droppati {n_before - len(df)} paziente/i senza follow-up (Time)")

print(f"\n{'═' * 65}")
print(f"  COMPLETATO — {df.shape[0]} pazienti × {df.shape[1]} feature")
print(f"{'═' * 65}")

df.to_csv(FILE_OUTPUT, index=False)
print(f"\n  Salvato in: {FILE_OUTPUT}")

═════════════════════════════════════════════════════════════════
  PIPELINE FROM RAW — oral cancer TCGA-HNSC (NO LEAKAGE)
═════════════════════════════════════════════════════════════════

[1/6] Caricamento file raw...
    Clinico: (2202, 25) | Follow-up: (1949, 6) | Rischio: (780, 9)

[2/6] Aggregazione per paziente al Tempo 0...
    Pazienti analizzati al Tempo 0: 528

[3/6] Follow-up: estrazione tempo di sopravvivenza...

[4/6] Integrazione Fattori di Rischio...

[5/6] Merge e Feature Encoding...

[6/6] Pulizia finale e salvataggio...
    Droppati 1 paziente/i senza follow-up (Time)

═════════════════════════════════════════════════════════════════
  COMPLETATO — 527 pazienti × 35 feature
═════════════════════════════════════════════════════════════════

  Salvato in: dataset.csv
